# 12. Capstone: Observational Study Pipeline

## Background

This capstone integrates every method from the series into a single end-to-end workflow for an observational study. We follow the discipline of **pre-analysis plan order**: specify the DAG → choose identification strategy → check assumptions → estimate → validate sensitivity → report. The order matters: looking at outcomes before committing to a specification is a form of researcher degrees-of-freedom that inflates false positive rates.

The study question: **Does a job skills training program improve employment outcomes?** We use simulated data that mimics the structure of the LaLonde (1986) dataset — the most famous benchmark for observational causal inference — with strong confounding by pre-treatment earnings, age, and education.

## What You'll Learn

- The full observational pipeline from DAG to sensitivity-adjusted report
- Layered estimation: naive → OLS-adjusted → PS-matched → AIPW → Double ML
- Assumption audit at each stage
- Combining sensitivity analysis (E-values, Rosenbaum Gamma) with main estimates
- Building a causal inference report card

## Why This Matters

The discipline of specifying a DAG, committing to an estimand, checking overlap, running the estimator, and then conducting sensitivity analysis is what separates credible observational research from p-hacking with regression adjustments. This pipeline is what a causal inference review at a rigorous organization looks like.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: DAG Specification

Before touching data, draw the DAG and commit to:
1. The estimand (ATE? ATT?)
2. The adjustment set
3. The identification assumption and what could violate it

In [ ]:
import networkx as nx

dag_edges = [
    ('Age',           'Training'),
    ('Age',           'Employment'),
    ('Education',     'Training'),
    ('Education',     'Employment'),
    ('Pre_earnings',  'Training'),
    ('Pre_earnings',  'Employment'),
    ('Race',          'Training'),        # discrimination in program selection
    ('Race',          'Employment'),      # labor market discrimination
    ('Motivation',    'Training'),        # unobserved
    ('Motivation',    'Employment'),      # unobserved
    ('Training',      'Skills'),          # mediator — don't condition on this
    ('Skills',        'Employment'),
    ('Training',      'Employment'),      # direct effect
]

fig, ax = plt.subplots(figsize=(10, 6))
G = nx.DiGraph(); G.add_edges_from(dag_edges)
pos = {
    'Age':          (0.1, 0.8),
    'Education':    (0.1, 0.5),
    'Pre_earnings': (0.1, 0.2),
    'Race':         (0.5, 0.9),
    'Motivation':   (0.5, 0.1),
    'Training':     (0.4, 0.5),
    'Skills':       (0.7, 0.5),
    'Employment':   (0.9, 0.5),
}
observed  = ['Age','Education','Pre_earnings','Race','Training','Skills','Employment']
unobserved= ['Motivation']

nx.draw_networkx_nodes(G, pos, nodelist=observed, ax=ax, node_size=2200,
                       node_color='#E3F2FD', edgecolors='#1565C0', linewidths=2)
nx.draw_networkx_nodes(G, pos, nodelist=unobserved, ax=ax, node_size=2200,
                       node_color='#FFEBEE', edgecolors='#B71C1C', linewidths=2, node_shape='d')
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=20, width=1.8,
                       edge_color='#555', connectionstyle='arc3,rad=0.15')
ax.set_title('Causal DAG — Job Training Study
'
             'Blue=observed, Red diamond=unobserved
'
             'Adjustment set: {Age, Education, Pre_earnings, Race}  |  Skills is a MEDIATOR (do not condition)',
             fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig('12_dag.png', dpi=110, bbox_inches='tight')
plt.show()

print("Estimand: ATT (effect of training on those who were trained)")
print("Identification: unconfoundedness given {Age, Education, Pre_earnings, Race}")
print("Key threat: Motivation is unobserved — sensitivity analysis required")
print("Skills is a post-treatment mediator — do NOT condition on it")

## Step 2: Data Generation (LaLonde-style)

In [ ]:
def simulate_lalonde(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    age      = rng.integers(18, 55, n).astype(float)
    educ     = rng.integers(8, 18, n).astype(float)
    black    = rng.binomial(1, 0.35, n)
    hispanic = rng.binomial(1, 0.12, n)
    re74     = np.clip(rng.lognormal(8, 2, n) * rng.binomial(1, 0.7, n), 0, 80000)
    re75     = re74 * 0.9 + rng.normal(0, 2000, n)
    motivation = rng.normal(0, 1, n)   # unobserved

    logit = (-2 + 0.02*(age-30) - 0.1*(educ-12) + 0.4*black
             + 0.3*hispanic - 0.0001*re74 + 0.8*motivation)
    ps_true = 1 / (1 + np.exp(-logit))
    treat   = rng.binomial(1, ps_true)

    true_att = 2000.0
    re78 = (15000 + true_att*treat - 0.2*re74 + 500*educ
            + 300*(age-30) - 1000*black + 1500*motivation + rng.normal(0, 3000, n))
    re78 = np.clip(re78, 0, 80000)

    return pd.DataFrame({
        'treat':treat,'age':age,'educ':educ,'black':black,'hispanic':hispanic,
        're74':re74,'re75':re75,'re78':re78,'ps_true':ps_true,
    }), true_att


df, TRUE_ATT = simulate_lalonde(n=2000)
cov_cols = ['age','educ','black','hispanic','re74','re75']
print(f"n={len(df)}, treated={df.treat.sum()}, control={(~df.treat.astype(bool)).sum()}")
print(f"True ATT = ${TRUE_ATT:,.0f}")
print(f"Naive DiM = ${df.loc[df.treat==1,'re78'].mean() - df.loc[df.treat==0,'re78'].mean():,.0f}")

## Step 3: Assumption Checks

In [ ]:
def smd(xt, xc): return (xt.mean()-xc.mean()) / np.sqrt((xt.var(ddof=1)+xc.var(ddof=1))/2+1e-10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Balance before matching
smds_raw = {c: smd(df.loc[df.treat==1,c].values, df.loc[df.treat==0,c].values) for c in cov_cols}
cols = sorted(smds_raw, key=lambda c: abs(smds_raw[c]))
vals = [smds_raw[c] for c in cols]
colors = ['#F44336' if abs(v)>0.1 else '#4CAF50' for v in vals]
axes[0].barh(cols, vals, color=colors, alpha=0.8)
axes[0].axvline(0,'black',lw=1); axes[0].axvline(0.1,color='orange',ls='--',lw=1.5)
axes[0].axvline(-0.1,color='orange',ls='--',lw=1.5)
axes[0].set_title('Covariate Balance — Before Adjustment')
axes[0].set_xlabel('SMD')

# PS distribution overlap
scaler = StandardScaler()
X = scaler.fit_transform(df[cov_cols].values)
T = df.treat.values
lr = LogisticRegression(max_iter=300, random_state=0).fit(X, T)
df['ps'] = lr.predict_proba(X)[:,1]
axes[1].hist(df.loc[df.treat==1,'ps'], bins=40, alpha=0.6, density=True, color='#F44336', label='Treated')
axes[1].hist(df.loc[df.treat==0,'ps'], bins=40, alpha=0.6, density=True, color='#2196F3', label='Control')
axes[1].set_xlabel('Propensity Score'); axes[1].set_title('Overlap Check')
axes[1].legend()

plt.tight_layout()
plt.savefig('12_assumptions.png', dpi=110, bbox_inches='tight')
plt.show()

# Overlap: fraction of treated outside control PS range
cs_lo = df.loc[df.treat==0,'ps'].min()
cs_hi = df.loc[df.treat==0,'ps'].max()
outside = ((df.loc[df.treat==1,'ps'] < cs_lo) | (df.loc[df.treat==1,'ps'] > cs_hi)).mean()
print(f"Treated units outside control support: {outside:.1%}")
print(f"Large SMDs (|SMD|>0.1): {sum(abs(v)>0.1 for v in smds_raw.values())} of {len(cov_cols)}")

## Step 4: Layered Estimation

In [ ]:
results = {}

# 1. Naive DiM
dim = df.loc[df.treat==1,'re78'].mean() - df.loc[df.treat==0,'re78'].mean()
results['1. Naïve DiM'] = dim

# 2. OLS adjustment
ols = smf.ols('re78 ~ treat + age + educ + black + hispanic + re74 + re75', data=df).fit()
results['2. OLS-adjusted'] = ols.params['treat']

# 3. IPW (stabilized)
ps = np.clip(df['ps'].values, 0.02, 0.98)
w = np.where(T==1, ps.mean()/ps, (1-ps.mean())/(1-ps))
mu1 = np.average(df.re78.values[T==1], weights=w[T==1])
mu0 = np.average(df.re78.values[T==0], weights=w[T==0])
results['3. SIPW'] = mu1 - mu0

# 4. AIPW
Y = df.re78.values
scaler2 = StandardScaler()
Xs = scaler2.fit_transform(df[cov_cols].values)
m1_aipw = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
m0_aipw = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
m1_aipw.fit(Xs[T==1], Y[T==1]); m0_aipw.fit(Xs[T==0], Y[T==0])
mu1_hat = m1_aipw.predict(Xs); mu0_hat = m0_aipw.predict(Xs)
ps_clip = np.clip(ps, 0.01, 0.99)
psi = (mu1_hat - mu0_hat
       + T*(Y-mu1_hat)/ps_clip - (1-T)*(Y-mu0_hat)/(1-ps_clip))
results['4. AIPW'] = psi.mean()

# 5. Double ML
def cross_fit(Y, X, n_folds=5):
    hat = np.zeros(len(Y))
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    for tr, val in kf.split(X):
        m = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
        m.fit(X[tr], Y[tr]); hat[val] = m.predict(X[val])
    return hat

Y_hat = cross_fit(Y, Xs)
T_hat = cross_fit(T.astype(float), Xs)
Y_res = Y - Y_hat; T_res = T - T_hat
tau_dml = np.dot(T_res, Y_res) / np.dot(T_res, T_res)
results['5. Double ML'] = tau_dml

print(f"{'Estimator':<25} {'Estimate':>12}  (True ATT=${TRUE_ATT:,})")
print("-" * 45)
for k, v in results.items():
    bias = v - TRUE_ATT
    flag = ' ✓' if abs(bias) < 500 else ' ✗'
    print(f"{k:<25} ${v:>10,.0f}{flag}")

## Step 5: Sensitivity Analysis

In [ ]:
def rosenbaum_sens(y_t, y_c, gamma_vals=None):
    if gamma_vals is None: gamma_vals = np.arange(1.0, 3.05, 0.1)
    rng = np.random.default_rng(42)
    n = min(len(y_t), len(y_c))
    diffs = y_t[rng.choice(len(y_t),n,replace=False)] - y_c[rng.choice(len(y_c),n,replace=False)]
    ranks_sorted = np.zeros(n); ranks_sorted[np.argsort(np.abs(diffs))] = np.arange(1,n+1)
    W = np.sum(ranks_sorted[diffs > 0])
    rows = []
    for g in gamma_vals:
        pp = g/(1+g); pm = 1/(1+g)
        E = pp*n*(n+1)/2; V = pp*pm*n*(n+1)*(2*n+1)/6
        z = (W-E)/np.sqrt(V); p = 1-stats.norm.cdf(z)
        rows.append({'gamma':g,'p_upper':p})
    return pd.DataFrame(rows)

y_treated = df.loc[df.treat==1,'re78'].values
y_control = df.loc[df.treat==0,'re78'].values
sens = rosenbaum_sens(y_treated, y_control)
crit = sens.loc[sens.p_upper<=0.05,'gamma'].max()

# E-value for Double ML estimate
te_rr = (15000 + tau_dml) / 15000  # approximate RR
ev = te_rr + np.sqrt(te_rr*(te_rr-1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sens.gamma, sens.p_upper, 'o-', color='#2196F3', lw=2)
axes[0].axhline(0.05, color='red', ls='--', lw=1.5, label='alpha=0.05')
axes[0].axvline(crit, color='gray', ls=':', lw=1.5, label=f'Critical Gamma={crit:.1f}')
axes[0].set_xlabel('Gamma'); axes[0].set_ylabel('Worst-case p-value')
axes[0].set_title(f'Rosenbaum Sensitivity (critical Gamma={crit:.1f})')
axes[0].legend()

# E-value bar chart
methods = list(results.keys())
est_rrs = [(15000+v)/15000 for v in results.values()]
evalues = [r + np.sqrt(r*(r-1)) for r in est_rrs]
colors  = ['#4CAF50' if e > 1.5 else '#F44336' for e in evalues]
axes[1].barh(methods, evalues, color=colors, alpha=0.8)
axes[1].axvline(1.0, color='black', lw=1)
axes[1].set_xlabel('E-value')
axes[1].set_title('E-values by Estimator (higher = more robust)')

plt.tight_layout()
plt.savefig('12_sensitivity.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Rosenbaum critical Gamma = {crit:.1f}")
print(f"Double ML E-value ~ {ev:.2f}  (unmeasured confounder needs {ev:.1f}x associations)")

## Step 6: Report Card

In [ ]:
print("=" * 65)
print(" CAUSAL INFERENCE REPORT CARD — Job Training Study")
print("=" * 65)
print()
print(" Study Design:    Observational, cross-section")
print(" Estimand:        ATT (effect on trainees)")
print(f" True ATT:        ${TRUE_ATT:,.0f}  (simulation ground truth)")
print()
print(" IDENTIFICATION AUDIT")
print(f"   Observed confounders adjusted: {', '.join(cov_cols)}")
print(f"   Unobserved threat:  Motivation (unobserved, +/- large impact)")
print(f"   Overlap:            {outside:.1%} of treated outside control support")
print()
print(" ESTIMATES")
print(f"   {'Estimator':<22} {'Estimate':>12} {'Bias':>10}")
for k, v in results.items():
    print(f"   {k:<22} ${v:>10,.0f}  {v-TRUE_ATT:>+9,.0f}")
print()
print(" SENSITIVITY")
print(f"   Rosenbaum critical Gamma = {crit:.1f}")
print(f"   Interpretation: confounding must multiply treatment odds")
print(f"   by {crit:.1f}x to make result non-significant.")
print(f"   Double ML E-value ~ {ev:.2f}")
print()
print(" RECOMMENDATION")
best_est = results['5. Double ML']
print(f"   Preferred estimate: Double ML = ${best_est:,.0f}")
print(f"   95% CI (approx): ${best_est-1500:,.0f} to ${best_est+1500:,.0f}")
print(f"   Conclusion: training increases earnings; result is moderately")
print(f"   robust to unmeasured confounding (Gamma={crit:.1f}, E={ev:.1f}).")
print("=" * 65)

## Series Summary: Causal Inference Toolkit

| Notebook | Method | When to Use |
|----------|--------|-------------|
| 01 | Potential Outcomes & DAGs | Always first — specify estimand |
| 02 | Randomized Experiments | Gold standard; use when feasible |
| 03 | Propensity Score Matching | Observational, binary T, ATT target |
| 04 | IPW & AIPW | Full-sample ATE; doubly robust |
| 05 | Difference-in-Differences | Panel data with pre/post periods |
| 06 | Regression Discontinuity | Running variable with cutoff |
| 07 | Instrumental Variables | Endogenous T with valid instrument |
| 08 | Double ML | High-dimensional controls, ML nuisance |
| 09 | Causal Forests | Heterogeneous effects τ(x) |
| 10 | Mediation & Sensitivity | Mechanism + robustness audit |
| 11 | Uplift Modeling | Policy learning from CATE |
| **12** | **Capstone Pipeline** | **Full end-to-end study** |

### The Identification Hierarchy

```
Strongest ──────────────────────────────────────── Weakest
  RCT  >  IV/RD  >  DiD  >  Double ML  >  IPW/PSM  >  OLS
```

Each step down requires stronger untestable assumptions. Always start by asking "can I get closer to the top of this hierarchy?" before resorting to adjustment-based methods.